In [20]:
import onnxmltools
import onnxruntime as rt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from skl2onnx.common.data_types import FloatTensorType
import numpy as np
import joblib

In [2]:
X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

In [3]:
sk_model = DecisionTreeClassifier(random_state=0)

sk_model.fit(X_train, y_train)

score = sk_model.score(X_test ,y_test)
print(f'Score: {score}')

Score: 0.98


In [21]:
joblib.dump(sk_model, 'decision_tree_iris.pkl')

['decision_tree_iris.pkl']

In [7]:
initial_type = [('float_input', FloatTensorType([None, 4]))]
onnx_model = onnxmltools.convert_sklearn(model=sk_model,
                                         name=sk_model.__class__.__name__,
                                         initial_types=initial_type,
                                         doc_string='MyDocstring',
                                         )

In [8]:
with open("decision_tree_iris.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [17]:
sess = rt.InferenceSession(
    "decision_tree_iris.onnx", providers=rt.get_available_providers())
input_name = sess.get_inputs()[0].name
pred_onx = sess.run(None, {input_name: X_test.astype(np.float32).to_numpy()})[0]
print(pred_onx)

[2 1 0 1 2 1 1 0 1 1 0 0 0 0 0 2 2 1 2 1 2 1 0 2 0 2 2 0 0 2 2 2 0 1 0 0 2
 1 2 1 1 1 0 0 2 1 2 2 1 2]


In [18]:
X_test_np = X_test.astype(np.float32).to_numpy()

sk_pred = sk_model.predict(X_test)
onnx_pred = sess.run(None, {input_name: X_test_np})[0]

print("Equal predictions:",
      np.array_equal(sk_pred, onnx_pred))

from sklearn.metrics import accuracy_score

print("Sklearn:", accuracy_score(y_test, sk_pred))
print("ONNX   :", accuracy_score(y_test, onnx_pred))

Equal predictions: True
Sklearn: 0.98
ONNX   : 0.98
